In [5]:
import sys, pathlib
ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / 'cartpole.py').exists() else pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import importlib, envs, evaluate as ev, cartpole as cp
importlib.reload(envs)
from controllers import lqr_discrete
import numpy as np

P = cp.CartPoleParams()
A, B = cp.analytic_AB(P)
dt = 0.02
A_d, B_d = cp.discretize(A, B, dt)
K_bry, _ = lqr_discrete(A_d, B_d, np.diag([4.0, 1.0, 25.0, 1.0]), np.array([[0.01]]), dt)

def bolum_oyna(env, politika, s0, seed):
    """Bir bölüm oynatır → (toplam ödül, uzunluk, terminated mı?)."""
    sonuc = env.reset(seed=seed, options={"s0": s0})
    if sonuc is None:
        raise RuntimeError("env.reset() returned None; check its implementation in envs.py.")
    o, _ = sonuc
    toplam, n, rng = 0.0, 0, np.random.default_rng(seed)
    while True:
        sonuc = env.step(politika(o, rng))
        if sonuc is None:
            raise RuntimeError("env.step() returned None; check its implementation in envs.py.")
        o, r, te, tr, _ = sonuc
        toplam += r; n += 1
        if te or tr:
            return toplam, n, te

politikalar = {
    "rastgele": lambda o, rng: rng.uniform(-1, 1, size=1).astype(np.float32),
    "sıfır":    lambda o, rng: np.zeros(1, dtype=np.float32),
    "LQR":      lambda o, rng: np.clip(-(K_bry @ o.astype(float)) / 10.0, -1, 1).astype(np.float32),
}
blok = ev.senaryolar("zor", 50, seed=0)
env = envs.CartPoleContinuousEnv()
print(f"{'politika':9s} | {'ort. ödül':>9s} | {'ort. uzunluk':>12s} | {'düştü':>6s}")
for ad, pol in politikalar.items():
    R = [bolum_oyna(env, pol, sc["s0"], sc["sensor_seed"]) for sc in blok]
    print(f"{ad:9s} | {np.mean([r[0] for r in R]):9.1f} | {np.mean([r[1] for r in R]):12.1f} | {sum(r[2] for r in R):3d}/50")

politika  | ort. ödül | ort. uzunluk |  düştü
rastgele  |       2.9 |         27.5 |  50/50
sıfır     |       4.0 |         29.6 |  50/50
LQR       |     384.2 |        426.3 |   8/50
